In [118]:
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
import json
from IPython.display import Markdown
from datasets import load_dataset
import requests


In [119]:
from huggingface_hub import login

hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    print("HuggingFace API key is not working")
else:
    print("HF KEY looks good so far")    
login(hf_token, add_to_git_credential=True)

HF KEY looks good so far


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [120]:
load_dotenv(override=True)
openai_api_key=os.getenv("OPENAI_API_KEY")
if not openai_api_key:
   print("OpenAI API Key not set properly")
else:
    print(f"OpenAI API Key exists")


openai=OpenAI()


OpenAI API Key exists


In [121]:
tools=[
    {
        "type": "function",
        "function": {
            "name": "about_stock",
            "description": "Get the full information about stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "stock_name": {
                        "type": "string",
                        "description": "The name of the stock",
                    },
                    "date":{
                        "type":"string",
                        "description":"Date of the trade for spesific stock"
                    }
                },
                "required": ["stock_name","date"],
                "additionalProperties": False
            }
        }
    },
    {
        "type":"function",
           "function":{
                "name":"about_news",
                "description":"News about any stocks",
                "parameters": {
                "type": "object",
                "properties": {
                    "stock_name": {
                        "type": "string",
                        "description": "The name of the stock",
                    },
                    "date":{
                        "type":"string",
                        "description":"Date of the news for spesific stock"
                    }
                },
                "required": ["stock_name","date"],
                "additionalProperties": False
              }
           }
    }
]

In [122]:
from massive import RESTClient
from datetime import datetime

def chech_date(date,format="%Y-%m-%d"):
    try:
        datetime.strptime(date,format)
        return True
    except ValueError:
        return False

massive_api_key=os.getenv("MASSIVE_API")
client = RESTClient(massive_api_key)

def about_stock(name,date):
    if chech_date(date)==True:
        request = client.get_daily_open_close_agg(
    name,
    date,
    adjusted="true",)
    else:
        print("Date is not avialable!Please check if format is valid")
    return request.close,request.open,request.high,request.low,request.volume
   

about_stock("AAPL","2026-02-11")



(275.5, 274.695, 280.18, 274.45, 51931283.0)

In [123]:

from massive.rest.models import (
    TickerNews,
)

def about_news(name,date):
   if chech_date(date)==True:
     news = []
     for n in client.list_ticker_news(
	  ticker=name.upper(),
	  published_utc=date,
	  order="asc",
	  limit="20",
	  sort="published_utc",
	  ):
      news.append(n)

      impr=[]
   for index, item in enumerate(news):
      n=[{"time":{item.published_utc},"publisher":{item.publisher.name},"url":{item.title},"URL":{item.article_url}}]
      impr.append(n)
   return impr
  

In [124]:
about_news("amzn","2026-02-23")

[[{'time': {'2026-02-23T03:30:00Z'},
   'publisher': {'The Motley Fool'},
   'url': {'Billionaire Value Investor Seth Klarman Sold Alphabet and Bought This Outstanding AI Stock Instead'},
   'URL': {'https://www.fool.com/investing/2026/02/22/billionaire-value-investor-seth-klarman-sold-alpha/?source=iedfolrf0000001'}}],
 [{'time': {'2026-02-23T14:30:00Z'},
   'publisher': {'The Motley Fool'},
   'url': {"The First 3 Stocks I'm Buying if the Market Crashes"},
   'URL': {'https://www.fool.com/investing/2026/02/23/the-first-3-stocks-im-buying-if-the-market-crashes/?source=iedfolrf0000001'}}],
 [{'time': {'2026-02-23T16:47:11Z'},
   'publisher': {'Benzinga'},
   'url': {'Walmart To Reward Corporate Staff With 121% Bonus Payout'},
   'URL': {'https://www.benzinga.com/markets/large-cap/26/02/50788629/walmart-to-reward-corporate-staff-with-121-bonus-payout?utm_source=benzinga_taxonomy&utm_medium=rss_feed_free&utm_content=taxonomy_rss&utm_campaign=channel'}}],
 [{'time': {'2026-02-23T16:53:00Z

In [129]:
system_prompt="""You are a helpfull assistant for "InvestAI" which helps people to give bunch of information about 
specific stocks,their price per share,total valume in market and give news sourse for each stock.Only if user asks information about any ticker(stock)
please make sure that user parses name of the stock and the date.You are not able to share information about 
price per share for today' market,however you can share news."""

In [126]:
def handle_tool_call(message):
    responses=[]

    for tool_call in message.tool_calls:
        func_name=tool_call.function.name
        args=json.loads(tool_call.function.arguments)
    
        if func_name=="about_stock":
         stock=args.get("stock_name")
         date=args.get("date")
         result=about_stock(stock.upper(),date)
        
        if func_name=="about_news":
            stock_name=args.get("stock_name")
            date_news=args.get("date")
            result=about_news(stock_name,date_news)

        content=json.dumps(result,ensure_ascii=False) if isinstance(result,dict) else str(result)

        responses.append({
            "role":"tool",
            "content":content,
            "tool_call_id":tool_call.id
         })

    return responses 


In [130]:
def chat_invest(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response=openai.chat.completions.create(
        model="gpt-5.4",
        messages=messages,
        tools=tools
    )
    
    while response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response=handle_tool_call(message)
        messages.append(message)
        messages.extend(response)
        response=openai.chat.completions.create(model="gpt-5.4", messages=messages)
       
    return response.choices[0].message.content


In [131]:
gr.ChatInterface(fn=chat_invest,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
